# Project Additional Materials — Spatial High Temperature Training

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  


**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:**  
1. First run `Spatial_dataset_prepare.ipynb` to generate the split datasets. All output files are saved in the `splits_mm/` folder, which must be at the same level as this notebook.  
2. Then run this notebook top to bottom in the same directory. It reads the prepared `.mm` files from `splits_mm/` and trains models on the spatial subsets.  

This notebook trains and evaluates models for the high-temperature spatial subsets (`top30`, `mid40`) using both CV and validation splits.  
The trained models and evaluation results are saved for reproducibility.  


## Step 0 — Import required libraries

In [3]:
import json, pickle, numpy as np
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## Step 1 — Load split datasets from `splits_mm/`
Load the prepared `.mm` files (top30, mid40) generated by `Spatial_dataset_prepare.ipynb`.  
These datasets are stored inside the `splits_mm/` folder at the same level as this notebook.


In [ ]:
# Load training (top30) and validation (mid40) splits
SEED = 42
DTYPE = np.dtype("float32")

# Function to load data splits
def load_split(prefix):
    with open(f"splits_mm/{prefix}_meta.json", "r") as f:
        meta = json.load(f)
    rows, cols = int(meta["rows"]), int(meta["cols"])
    X = np.memmap(f"splits_mm/{prefix}_X.mm", mode="r", dtype=DTYPE, shape=(rows, cols))
    y = np.memmap(f"splits_mm/{prefix}_y.mm", mode="r", dtype=DTYPE, shape=(rows,))
    return np.asarray(X), np.asarray(y), meta.get("features", None)

# Function to print evaluation metrics
def print_metrics(tag, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    loss = 1.0 - r2
    print(f"[{tag}] R2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, loss(1-R2)={loss:.4f}")
    return r2, rmse, mae, loss

# top30 for training
X_train, y_train, feats_train = load_split("top30")
# mid40 for validation (only needed in Version 3)
X_val, y_val, feats_val = load_split("mid40")

## Step 2 — Train Baseline AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_high2low_cv.pkl`.

In [ ]:
# FLAML training with CV (top30 only)
automl = AutoML()
automl.fit(
    X_train, y_train,
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="cv", # cross-validation
    n_splits=5, # number of CV splits
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on training and validation sets
print("[CV] Best estimator:", automl.best_estimator)
print("[CV] Best config:", automl.best_config)
print("[CV] Best CV loss (1 - R2):", automl.best_loss)

# save model
model_save_path = "automl_high2low_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[CV] Model saved as {model_save_path}")

[CV] Best estimator: catboost
[CV] Best config: {'early_stopping_rounds': 10, 'learning_rate': 0.09999999999999996, 'n_estimators': 8192}
[CV] Best CV loss (1 - R2): 0.018584352309085617
[CV] Model saved as automl_high2low_cv.pkl


## Step 3 — Train Baseline AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_high2low_val.pkl`.

In [ ]:
# FLAML training with explicit validation (top30 train, mid40 val)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    X_val=X_val, y_val=y_val,   # external validation
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="holdout", # holdout validation
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on validation set (mid40 only)
print("[VAL] Best estimator:", automl.best_estimator)
print("[VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_high2low_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[VAL] Model saved as {model_save_path}")


[VAL] Best estimator: rf
[VAL] Best config: {'n_estimators': 65, 'max_features': 0.30712644795304467, 'max_leaves': 1583}
[VAL(mid40)] R2=0.9314, RMSE=1.3111, MAE=1.0242, loss(1-R2)=0.0686
[VAL] Model saved as automl_high2low_val.pkl


## Step 4 — Train Random Forest AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_rf_high2low_cv.pkl`.

In [ ]:
# RF + CV (top30 only)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="cv", # cross-validation
    n_splits=5, # number of CV splits
    estimator_list=["rf"], # only use random forest
    n_jobs=-1, # use all CPUs
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on training and validation sets
print("[RF-CV] Best estimator:", automl.best_estimator)
print("[RF-CV] Best config:", automl.best_config)
print("[RF-CV] Best CV loss (1 - R2):", automl.best_loss)

# save model
model_save_path = "automl_rf_high2low_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[RF-CV] Model saved as {model_save_path}")

[RF-CV] Best estimator: rf
[RF-CV] Best config: {'n_estimators': 4, 'max_features': 1.0, 'max_leaves': 29}
[RF-CV] Best CV loss (1 - R2): 0.07417692101795581
[RF-CV] Model saved as automl_rf_high2low_cv.pkl


## Step 5 — Train Random Forest AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_rf_high2low_val.pkl`.

In [ ]:
# RF + explicit validation (mid40)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    X_val=X_val, y_val=y_val, # external validation
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="holdout", # holdout validation
    estimator_list=["rf"], # only use random forest
    n_jobs=-1, # use all CPUs
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on validation set (mid40 only)
print("[RF-VAL] Best estimator:", automl.best_estimator)
print("[RF-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_rf_high2low_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[RF-VAL] Model saved as {model_save_path}")


[RF-VAL] Best estimator: rf
[RF-VAL] Best config: {'n_estimators': 65, 'max_features': 0.30712644795304467, 'max_leaves': 1583}
[VAL(mid40)] R2=0.9314, RMSE=1.3111, MAE=1.0242, loss(1-R2)=0.0686
[RF-VAL] Model saved as automl_rf_high2low_val.pkl


## Step 6 — Train XGBoost AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_xgb_high2low_cv.pkl`.

In [ ]:
# XGBoost + CV (top30 only)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="cv", # cross-validation
    n_splits=5, # number of CV splits
    estimator_list=["xgboost"], # only use XGBoost
    n_jobs=-1, # use all CPUs
    seed=42,# random seed
    verbose=1,# print training progress
)

# Evaluate on training and validation sets
print("[XGB-CV] Best estimator:", automl.best_estimator)
print("[XGB-CV] Best config:", automl.best_config)
print("[XGB-CV] Best CV loss (1 - R2):", automl.best_loss)

# save model
model_save_path = "automl_xgb_high2low_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[XGB-CV] Model saved as {model_save_path}")


[XGB-CV] Best estimator: xgboost
[XGB-CV] Best config: {'n_estimators': 1109, 'max_leaves': 18, 'min_child_weight': 0.40385496411102617, 'learning_rate': 0.0951546340177734, 'subsample': 0.7621325607358561, 'colsample_bylevel': 0.896142769508154, 'colsample_bytree': 0.9993271961638156, 'reg_alpha': 0.0014585172191691578, 'reg_lambda': 17.50258170562381}
[XGB-CV] Best CV loss (1 - R2): 0.018307386643973668
[XGB-CV] Model saved as automl_xgb_high2low_cv.pkl


## Step 7 — Train XGBoost AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_xgb_high2low_val.pkl`.

In [ ]:
# XGBoost + explicit validation (mid40)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    X_val=X_val, y_val=y_val, # external validation
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="holdout", # holdout validation
    estimator_list=["xgboost"], # only use XGBoost
    n_jobs=-1, # use all CPUs
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on validation set (mid40 only)
print("[XGB-VAL] Best estimator:", automl.best_estimator)
print("[XGB-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_xgb_high2low_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[XGB-VAL] Model saved as {model_save_path}")


[XGB-VAL] Best estimator: xgboost
[XGB-VAL] Best config: {'n_estimators': 40, 'max_leaves': 5, 'min_child_weight': 0.8296298264591767, 'learning_rate': 0.1034117518337477, 'subsample': 0.7832902196547024, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9384436197975786, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.0055053933160862595}
[VAL(mid40)] R2=0.9187, RMSE=1.4264, MAE=1.1390, loss(1-R2)=0.0813
[XGB-VAL] Model saved as automl_xgb_high2low_val.pkl


## Step 8 — Train LightGBM AutoML model with FLAML (CV)
Train the models and select the best one to save `automl_lgbm_high2low_cv.pkl`.

In [ ]:
# LightGBM + CV (top30 only)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="cv", # cross-validation
    n_splits=5, # number of CV splits
    estimator_list=["lgbm"], # only use LightGBM
    n_jobs=-1, # use all CPUs
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on training and validation sets
print("[LGBM-CV] Best estimator:", automl.best_estimator)
print("[LGBM-CV] Best config:", automl.best_config)
print("[LGBM-CV] Best CV loss (1 - R2):", automl.best_loss)

# save model
model_save_path = "automl_lgbm_high2low_cv.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[LGBM-CV] Model saved as {model_save_path}")

[LGBM-CV] Best estimator: lgbm
[LGBM-CV] Best config: {'n_estimators': 1921, 'num_leaves': 51, 'min_child_samples': 11, 'learning_rate': 1.0, 'log_max_bin': 6, 'colsample_bytree': 1.0, 'reg_alpha': 0.021120634578611474, 'reg_lambda': 1.0995349168568005}
[LGBM-CV] Best CV loss (1 - R2): 0.019483659157973454
[LGBM-CV] Model saved as automl_lgbm_high2low_cv.pkl


## Step 9 — Train LightGBM AutoML model with FLAML (Mid40)
Train the models and select the best one to save `automl_lgbm_high2low_val.pkl`.

In [ ]:
# LightGBM + explicit validation (mid40)
automl = AutoML()
automl.fit(
    X_train, y_train, # training data
    X_val=X_val, y_val=y_val, # external validation
    task="regression", # regression task
    metric="r2", # use R2 as the metric
    time_budget=3600, # total running time in seconds
    eval_method="holdout", # holdout validation
    estimator_list=["lgbm"], # only use LightGBM
    n_jobs=-1, # use all CPUs
    seed=42, # random seed
    verbose=1, # print training progress
)

# Evaluate on validation set (mid40 only)
print("[LGBM-VAL] Best estimator:", automl.best_estimator)
print("[LGBM-VAL] Best config:", automl.best_config)

# predict on validation set
y_val_pred = automl.predict(X_val)
print_metrics("VAL(mid40)", y_val, y_val_pred)

# save model
model_save_path = "automl_lgbm_high2low_val.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(automl, f, pickle.HIGHEST_PROTOCOL)
print(f"[LGBM-VAL] Model saved as {model_save_path}")


[LGBM-VAL] Best estimator: lgbm
[LGBM-VAL] Best config: {'n_estimators': 23, 'num_leaves': 9, 'min_child_samples': 8, 'learning_rate': 0.18841855764236484, 'log_max_bin': 6, 'colsample_bytree': 0.7801445636679567, 'reg_alpha': 0.00958475986897041, 'reg_lambda': 0.024701768807523516}
[VAL(mid40)] R2=0.9220, RMSE=1.3978, MAE=1.1034, loss(1-R2)=0.0780
[LGBM-VAL] Model saved as automl_lgbm_high2low_val.pkl
